# SASRec + TwoTower 학습 노트북

이 노트북은 **artwork_vector.json**(작품 임베딩)과 **train_user_logs.json/jsonl**(유저 로그)를 이용해서  
`FeatureSASRec` + `TwoTowerAlign`(cosine/temperature scaling) 을 학습합니다.

- PAD index = 0, 실제 아이템은 1부터 시작
- 로그는 user별 시퀀스로 모아서 학습합니다.
- 평가: Leave-one-out (각 유저의 마지막 아이템 맞추기) HR/NDCG


In [36]:
# Cell 1) Config (경로/하이퍼파라미터만 여기서 수정)

from pathlib import Path

# 현재 노트북 실행 위치(=주피터의 working directory)를 기준으로 경로 잡기
BASE_DIR = Path.cwd()

# 입력 파일
LOG_PATH = str(BASE_DIR / "outputs_json" / "user_logs.json")  # 또는 .jsonl
VEC_PATH = str(BASE_DIR / "outputs_json" / "artwork_vector.json")   # 작품 벡터 JSON

# 출력 폴더 (없으면 생성)
OUT_PTH_DIR = BASE_DIR / "outputs_pth"
OUT_PTH_DIR.mkdir(parents=True, exist_ok=True)

# 저장 파일
OUT_SASREC = str(OUT_PTH_DIR / "BEST_SASRec_model.pth")
OUT_TWOTOWER = str(OUT_PTH_DIR / "Best_UserRecommend_model.pth")

# 모델/학습 하이퍼파라미터
MAX_LEN = 200
HIDDEN_DIM = 512

BATCH_SIZE = 256
EPOCHS = 30
LR = 1e-3
SEED = 42

N_LAYERS = 2
N_HEADS = 4
DROPOUT = 0.1

LOGIT_SCALE = 20.0  # temperature scaling (bigger => sharper)

print("LOG_PATH:", LOG_PATH)
print("VEC_PATH:", VEC_PATH)
print("OUT_SASREC:", OUT_SASREC)
print("OUT_TWOTOWER:", OUT_TWOTOWER)

LOG_PATH: /home/j-i14e107/Image_classification/outputs_json/user_logs.json
VEC_PATH: /home/j-i14e107/Image_classification/outputs_json/artwork_vector.json
OUT_SASREC: /home/j-i14e107/Image_classification/outputs_pth/BEST_SASRec_model.pth
OUT_TWOTOWER: /home/j-i14e107/Image_classification/outputs_pth/Best_UserRecommend_model.pth


In [37]:
# Cell 2) Imports
import json, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from tqdm import tqdm


In [38]:
# Cell 3) Utils
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def read_json_or_jsonl(path: str):
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(f"File not found: {path}")
    txt = p.read_text(encoding="utf-8").strip()
    if not txt:
        raise ValueError(f"Empty file: {path}")
    if txt[0] == "{" and "\n" in txt:
        lines = [ln.strip() for ln in txt.splitlines() if ln.strip()]
        ok_jsonl = all(ln.startswith("{") and ln.endswith("}") for ln in lines[: min(5, len(lines))])
        if ok_jsonl:
            return [json.loads(ln) for ln in lines]
    return json.loads(txt)

def stable_hash_vector(vec: list) -> str:
    arr = np.asarray(vec, dtype=np.float32)
    arr = np.round(arr, 3)
    return "vec_" + str(abs(hash(arr.tobytes())))

set_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
device


'cuda'

In [39]:
# Cell: FeatureSASRec (FIXED, trainable item_emb + Transformer)

import torch
import torch.nn as nn
import torch.nn.functional as F

class FeatureSASRec(nn.Module):
    def __init__(self, item_vectors: torch.Tensor, hidden_dim=512, n_layers=2, n_heads=4, dropout=0.1, maxlen=200):
        super().__init__()

        # item_vectors: (num_items, embed_dim) with padding at index 0
        assert item_vectors.dim() == 2, "item_vectors must be (N, D)"
        num_items, embed_dim = item_vectors.shape

        # ✅ 학습 가능한 item embedding
        self.item_emb = nn.Embedding(num_items, embed_dim, padding_idx=0)
        with torch.no_grad():
            self.item_emb.weight.copy_(item_vectors)

        # (선택) 원본 벡터를 buffer로도 저장(학습 안 됨)
        self.register_buffer("item_vectors", item_vectors.clone())

        # position embedding
        self.pos_emb = nn.Embedding(maxlen, embed_dim)

        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=n_heads,
            dim_feedforward=4 * embed_dim,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)

        self.dropout = nn.Dropout(dropout)
        self.maxlen = maxlen
        self.embed_dim = embed_dim

    @property
    def item_vectors_trainable(self):
        # 학습되는 item embedding weight
        return self.item_emb.weight

    def forward(self, input_ids: torch.Tensor):
        """
        input_ids: (B, S) with 0 padding
        returns: (B, S, D)
        """
        B, S = input_ids.shape
        assert S <= self.maxlen, f"seq len {S} > maxlen {self.maxlen}"

        # item embedding
        x = self.item_emb(input_ids)  # (B,S,D)

        # position embedding
        pos = torch.arange(S, device=input_ids.device).unsqueeze(0).expand(B, S)
        x = x + self.pos_emb(pos)

        x = self.dropout(x)

        # padding mask: True where PAD
        pad_mask = (input_ids == 0)
        x = self.encoder(x, src_key_padding_mask=pad_mask)

        return x

In [40]:
# Cell 5) Data loaders
from pathlib import Path

def _stem_id(x) -> str:
    # "category096_0001.png" 또는 "/.../category096_0001.png" -> "category096_0001"
    return Path(str(x)).stem

def load_item_vectors(vec_json_path: str, expected_dim: int = 512):
    data = read_json_or_jsonl(vec_json_path)

    artwork2idx = {"<PAD>": 0}
    matrix_list = [np.zeros(expected_dim, dtype=np.float32)]

    def add_one(aid, vec):
        if aid is None or vec is None:
            return
        if isinstance(vec, list) and len(vec) == expected_dim:
            aid = _stem_id(aid)  # ✅ id 규칙 통일 (확장자/경로 제거)
            if aid not in artwork2idx:
                artwork2idx[aid] = len(artwork2idx)
                matrix_list.append(np.array(vec, dtype=np.float32))

    if isinstance(data, dict):
        for aid, vec in data.items():
            add_one(aid, vec)
    elif isinstance(data, list):
        for item in data:
            if not isinstance(item, dict):
                continue
            aid = item.get("artwork_id") or item.get("item_id") or item.get("id")
            vec = item.get("artwork_vector") or item.get("vector") or item.get("embedding")
            add_one(aid, vec)
    else:
        raise ValueError("Unsupported vector JSON format")

    item_mat = torch.tensor(np.stack(matrix_list), dtype=torch.float32)
    item_mat = item_mat / (item_mat.norm(dim=-1, keepdim=True) + 1e-12)
    return artwork2idx, item_mat

def load_user_sequences(log_path: str, artwork2idx: dict, logs_are_latest_first: bool = True):
    """유저 로그는 이제 (member_id, artwork_id, action/timestamp)만 있으면 됨.
    - artwork_id는 stem 규칙으로 통일해서 벡터 키와 매칭
    - 로그가 최신순(1줄=가장 최신)이면 학습용(과거->현재)으로 reverse
    """
    logs = read_json_or_jsonl(log_path)
    if not isinstance(logs, list):
        raise ValueError("Train log must be JSON array or JSONL list")

    user_seq = {}
    missed = 0
    for log in logs:
        if not isinstance(log, dict):
            continue

        uid = str(log.get("member_id"))
        aid = log.get("artwork_id")

        if aid is None:
            missed += 1
            continue

        aid = _stem_id(aid)  # ✅ id 규칙 통일
        if aid in artwork2idx:
            user_seq.setdefault(uid, []).append(artwork2idx[aid])
        else:
            missed += 1

    if logs_are_latest_first:
        for uid in user_seq:
            user_seq[uid] = list(reversed(user_seq[uid]))

    return user_seq, missed


In [41]:
# Cell 6) Dataset
class SASRecDataset(Dataset):
    def __init__(self, user_seq, maxlen=200):
        self.samples = []
        for seq in user_seq.values():
            if len(seq) < 2:
                continue
            if len(seq) > maxlen + 1:
                seq = seq[-(maxlen + 1):]
            input_ids = seq[:-1]
            target_ids = seq[1:]
            pad_len = maxlen - len(input_ids)
            input_ids = [0] * pad_len + input_ids
            target_ids = [0] * pad_len + target_ids
            self.samples.append((torch.tensor(input_ids), torch.tensor(target_ids)))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]


In [42]:
# Cell 7) Evaluation
@torch.no_grad()
def evaluate_valid(sas_model, tt_model, user_seq, all_item_vecs, maxlen=200, device='cuda'):
    sas_model.eval()
    tt_model.eval()

    HR_10, HR_20, NDCG_10, NDCG_20 = [], [], [], []

    all_items_proj = tt_model.item_proj(all_item_vecs.to(device))
    all_items_proj = F.normalize(all_items_proj, p=2, dim=-1)

    users = [u for u in user_seq.keys() if len(user_seq[u]) >= 2]
    for uid in tqdm(users, desc="eval", leave=False):
        seq = user_seq[uid]
        input_seq = seq[:-1]
        target_item = seq[-1]
        if len(input_seq) == 0:
            continue

        if len(input_seq) > maxlen:
            input_seq = input_seq[-maxlen:]
        pad_len = maxlen - len(input_seq)
        input_tensor = torch.tensor(([0] * pad_len + input_seq), device=device).unsqueeze(0)

        sas_out = sas_model(input_tensor)
        last_emb = sas_out[:, -1, :]

        user_vec = tt_model.user_proj(last_emb)
        user_vec = F.normalize(user_vec, p=2, dim=-1)

        scores = torch.matmul(user_vec, all_items_proj.T).squeeze(0)
        scores[0] = -1e9

        _, top_indices = torch.topk(scores, k=20)
        top_indices = top_indices.detach().cpu().numpy()

        hit_10 = 1 if target_item in top_indices[:10] else 0
        hit_20 = 1 if target_item in top_indices[:20] else 0
        HR_10.append(hit_10)
        HR_20.append(hit_20)

        ndcg_10 = 0.0
        ndcg_20 = 0.0
        if hit_10:
            rank = np.where(top_indices[:10] == target_item)[0][0]
            ndcg_10 = 1.0 / np.log2(rank + 2)
        if hit_20:
            rank = np.where(top_indices[:20] == target_item)[0][0]
            ndcg_20 = 1.0 / np.log2(rank + 2)
        NDCG_10.append(ndcg_10)
        NDCG_20.append(ndcg_20)

    return float(np.mean(HR_10)), float(np.mean(HR_20)), float(np.mean(NDCG_10)), float(np.mean(NDCG_20))

@torch.no_grad()
def count_hits_at20(sas_model, tt_model, user_seq, all_item_vecs, maxlen=200, device='cuda'):
    sas_model.eval(); tt_model.eval()
    all_items_proj = F.normalize(tt_model.item_proj(all_item_vecs.to(device)), p=2, dim=-1)

    users = [u for u in user_seq.keys() if len(user_seq[u]) >= 2]
    hits = 0
    n = 0
    for uid in tqdm(users, desc="eval_hitcount", leave=False):
        seq = user_seq[uid]
        input_seq = seq[:-1]
        target_item = seq[-1]
        if len(input_seq) == 0:
            continue

        if len(input_seq) > maxlen:
            input_seq = input_seq[-maxlen:]
        pad_len = maxlen - len(input_seq)
        x = torch.tensor(([0]*pad_len + input_seq), device=device).unsqueeze(0)

        last_emb = sas_model(x)[:, -1, :]
        user_vec = F.normalize(tt_model.user_proj(last_emb), p=2, dim=-1)
        scores = (user_vec @ all_items_proj.T).squeeze(0)
        scores[0] = -1e9
        top20 = torch.topk(scores, k=20).indices.detach().cpu().numpy()

        hits += int(target_item in top20)
        n += 1

    print("users evaluated:", n)
    print("hits@20:", hits)
    print("HR@20:", hits / max(n, 1))
    return hits, n

In [43]:
# Cell 8) Load data
artwork2idx, item_mat = load_item_vectors(VEC_PATH, expected_dim=512)
user_seq, missed = load_user_sequences(LOG_PATH, artwork2idx, logs_are_latest_first=True)

print(f"items={len(artwork2idx)-1} users={len(user_seq)} missed_logs={missed}")
print("item_mat:", item_mat.shape)

dataset = SASRecDataset(user_seq, maxlen=MAX_LEN)
print("train samples:", len(dataset))

loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)

print('missed ratio:', missed / max(len(read_json_or_jsonl(LOG_PATH)), 1))


items=5000 users=20000 missed_logs=0
item_mat: torch.Size([5001, 512])
train samples: 19472
missed ratio: 0.0


In [44]:
# Cell 9) Build models + optimizer
sas_model = FeatureSASRec(
    item_vectors=item_mat.to(device),
    hidden_dim=HIDDEN_DIM,
    n_layers=N_LAYERS,
    n_heads=N_HEADS,
    dropout=DROPOUT,
    maxlen=MAX_LEN,
).to(device)

tt_model = TwoTowerAlign(dim=HIDDEN_DIM, dropout=DROPOUT).to(device)

params = list(sas_model.parameters()) + list(tt_model.parameters())
optimizer = torch.optim.Adam(params, lr=LR)
loss_fn = nn.CrossEntropyLoss(ignore_index=0)

best_hr10 = -1.0
best_hr20 = -1.0

In [ ]:
# (Cell 10 상단 어딘가) best 변수 초기화
best_hr20 = -1.0  # 또는 0.0
best_hr10 = -1.0  # (원하면 같이 유지)

# Cell 10) Train loop
for epoch in range(1, EPOCHS + 1):
    sas_model.train()
    tt_model.train()
    total_loss = 0.0

    pbar = tqdm(loader, desc=f"Epoch {epoch}/{EPOCHS}")
    for input_ids, target_ids in pbar:
        input_ids = input_ids.to(device)
        target_ids = target_ids.to(device)

        sas_emb = sas_model(input_ids) # (B,S,512)
        user_final = tt_model.user_proj(sas_emb.reshape(-1, H))
        user_final = F.normalize(user_final, p=2, dim=-1)
        B, S, H = sas_emb.shape

        user_final = tt_model.user_proj(sas_emb.reshape(-1, H))
        user_final = F.normalize(user_final, p=2, dim=-1)

        all_items_vec = sas_model.item_vectors_trainable
        item_final = tt_model.item_proj(sas_model.item_vectors_trainable)
        item_final = F.normalize(item_final, p=2, dim=-1)

        logits = torch.matmul(user_final, item_final.T)
        logits[:, 0] = -1e9
        logits = logits * LOGIT_SCALE

        targets = target_ids.reshape(-1)
        mask = targets != 0  # PAD 위치 제거
        loss = loss_fn(logits[mask], targets[mask])

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += float(loss.item())
        pbar.set_postfix(loss=float(loss.item()))

    avg_loss = total_loss / max(len(loader), 1)
    print(f"[Epoch {epoch}] avg_loss={avg_loss:.4f} | evaluating...")

    hr10, hr20, ndcg10, ndcg20 = evaluate_valid(sas_model, tt_model, user_seq, item_mat, MAX_LEN, device)
    print(f"   HR@10={hr10:.4f} HR@20={hr20:.4f} NDCG@10={ndcg10:.4f} NDCG@20={ndcg20:.4f}")

    # ✅ 저장 기준을 HR@20으로 변경
    if hr20 > best_hr20:
        best_hr20 = hr20
        best_hr10 = max(best_hr10, hr10)  # (선택) 기록용
        print(f"   [Best] saving checkpoints (best_hr20={best_hr20:.4f})")

        torch.save(
            {"state_dict": sas_model.state_dict(),
             "config": {"hidden": HIDDEN_DIM, "maxlen": MAX_LEN,
                        "n_layers": N_LAYERS, "n_heads": N_HEADS, "dropout": DROPOUT}},
            OUT_SASREC
        )
        torch.save(
            {"two_tower_state_dict": tt_model.state_dict(),
             "config": {"hidden": HIDDEN_DIM, "dropout": DROPOUT}},
            OUT_TWOTOWER
        )

print("Done. best_hr20:", best_hr20, "| best_hr10:", best_hr10)

Epoch 1/30: 100%|██████████| 77/77 [00:11<00:00,  6.85it/s, loss=7.77]


[Epoch 1] avg_loss=7.8181 | evaluating...


   HR@10=0.0259 HR@20=0.0485 NDCG@10=0.0180 NDCG@20=0.0238
   [Best] saving checkpoints (best_hr20=0.0485)


Epoch 2/30: 100%|██████████| 77/77 [00:11<00:00,  6.94it/s, loss=7.47]


[Epoch 2] avg_loss=7.7449 | evaluating...


...

[Epoch 3] avg_loss=7.7438 | evaluating...


   HR@10=0.0262 HR@20=0.0402 NDCG@10=0.0172 NDCG@20=0.0207


Epoch 4/30: 100%|██████████| 77/77 [00:11<00:00,  6.92it/s, loss=7.77]


[Epoch 4] avg_loss=7.7191 | evaluating...


eval:  57%|█████▋    | 11017/19472 [00:09<00:07, 1138.69it/s]